In [1]:
import os
import sys

import pandas as pd
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import joblib

from sklearnex import patch_sklearn
patch_sklearn()

import config as cfg

TARGET = 'daylight-2016-01-06T00_00_00Z'

Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


# Daytime Creation

In [3]:
daytime_model = joblib.load(os.path.join('.', 'models__', 'DayTimeClustering_7Clusters.joblib'))
daytime_model

KMeans(n_clusters=7)

In [11]:
import re

grids = {}
dir_path = os.path.join(cfg.IMG_DATA_PATH, 'daylight-30m-per-pixel', '2016')
for grid in os.listdir(dir_path):
    if grid == '.DS_Store': continue
    if not grid.startswith(TARGET): continue

    img = cv.imread(os.path.join(dir_path, grid))
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

    grid_num = int(re.findall(f'{TARGET}_(.+)\.tiff', grid)[0])

    grids[grid_num] = img
    print(f'Added GRID_NUM {grid_num}', end='\r')


In [4]:
test = pd.DataFrame(grids[0].reshape(grids[0].shape[0] * grids[1].shape[1], 3), columns=['R','G','B'])

for col in test.columns:
    test[col] = test[col] / 255.

test['cluster'] = daytime_model.predict(test)

test

,R,G,B,cluster
0,0.023529,0.058824,0.086275,6
1,0.023529,0.058824,0.086275,6
2,0.023529,0.058824,0.086275,6
3,0.023529,0.058824,0.086275,6
4,0.023529,0.058824,0.086275,6
...,...,...,...,...
12415,0.023529,0.058824,0.086275,6
12416,0.023529,0.058824,0.086275,6
12417,0.023529,0.058824,0.086275,6
12418,0.023529,0.058824,0.086275,6


In [5]:
cluster_maps = {
    'cloud': [1, 4, 5],
    'landmass': [0, 2, 3],
    'water': [6]
}

test.groupby('cluster').count().reset_index().where(test['cluster'].isin(cluster_maps['water']))['R'].sum()

12420

In [6]:
# Image Processing

cluster_maps = {
    'cloud': [1, 4, 5],
    'landmass': [0, 2, 3],
    'water': [6]
}


data = {
    'grid_num': [],
    'cloud_percentage': [],
    'landmass_percentage': [],
    'water_percentage': []
}

def calculate_feature_percentage(grid, feature):
    num_pixels = grid.shape[0]

    feature_count = grid.groupby('cluster')\
                        .count()\
                        .reset_index()\
                        .where(grid['cluster'].isin(cluster_maps[feature]))['R']\
                        .sum()
    
    return feature_count / num_pixels



for grid_num, grid in grids.items():
    reshaped_grid = pd.DataFrame(grid.reshape(grid.shape[0] * grid.shape[1], 3), columns=['R','G','B'])
    
    for col in reshaped_grid.columns:
        reshaped_grid[col] = reshaped_grid[col] / 255.

    reshaped_grid['cluster'] = daytime_model.predict(reshaped_grid)

    # cloud_density - Number of pixels in cluster 1, 4, 5
    data['grid_num'].append(grid_num)

    for feat in ['cloud', 'landmass', 'water']:
        data[f'{feat}_percentage'].append(calculate_feature_percentage(reshaped_grid, feat))
    
    print(f'Done Encoding Data {grid_num} to Tabular', end='\r')

recom_pool = pd.DataFrame(data).sort_values(by='grid_num').set_index('grid_num')

recom_pool
    

,cloud_percentage,landmass_percentage,water_percentage
grid_num,,,
0,0.0,0.0,1.0
1,0.0,0.0,1.0
2,0.0,0.0,1.0
3,0.0,0.0,1.0
4,0.0,0.0,1.0
...,...,...,...
2495,0.0,1.0,0.0
2496,0.0,1.0,0.0
2497,0.0,1.0,0.0


In [7]:
recom_pool.where((recom_pool['water_percentage'] > 0) & (recom_pool['water_percentage'] < 1)).dropna()

,cloud_percentage,landmass_percentage,water_percentage
grid_num,,,
363,0.0,0.588003,0.411997
378,0.0,0.177456,0.822544
412,0.0,0.213205,0.786795
519,0.0,0.590660,0.409340
865,0.0,0.005233,0.994767
896,0.0,0.678905,0.321095
1072,0.0,0.804187,0.195813
1215,0.0,0.792029,0.207971
1366,0.0,0.884058,0.115942


# Calculating Light Activity

In [3]:
nighttime_model = joblib.load(os.path.join('.', 'models', 'NightTimeClustering_3Clusters.joblib'))
nighttime_model

KMeans(n_clusters=3)

In [11]:
from PIL import Image

In [4]:
TARGET = 'r2-30m-snapshot-2016-11-30T00_00_00Z'

In [13]:
import re

grids = {}
dir_path = os.path.join(cfg.IMG_DATA_PATH, 'night-30m-per-pixel', '2016')
for grid in os.listdir(dir_path):
    if grid == '.DS_Store': continue
    if not grid.startswith(TARGET): continue
    img = Image.open(os.path.join(dir_path, grid))

    img = cv.imread(os.path.join(dir_path, grid))
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

    grid_num = int(re.findall(f'{TARGET}_(.+)\.tiff', grid)[0])

    grids[grid_num] = img
    print(f'Added GRID_NUM {grid_num}', end='\r')

grids

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\paolo\\OneDrive\\Desktop\\Projects\\laguna-philippines-mapping-fisheries\\src\\tasks\\task-3-recommendation-engine\\subtask-2-image-processing\\..\\..\\..\\data\\Satellite Images\\Processed Images\\night-30m-per-pixel\\2016\\r2-30m-snapshot-2016-11-30T00_00_00Z_0.tiff'

In [49]:
grids

{}

# Adding Tabular Instances